## DSAI4205 Big Data Analytics
## Assignment 2 (10%), Term 2, 2025-2026

#### **Instructions for the assignment**:
* Follow the provided instructions to solve the questions.
* Please save the report and named as DSAI4205Assignment2_{your student_id}.ipynb (e.g., if student_id is 1234567, then the file's name is DSAI4205Assignment2_1234567.ipynb).

* Submit the notebook file to the blackboard. The submission due date is **<font color='red'>24 Mar 2026 23:55</font>**

<font color='red'>**You should complete the assignment on your own. You must uphold academic integrity and academic honesty in the assignment. You can be subject to disciplinary action if proven to have acted against academic integrity and academic honesty.**</font>

### Question 1 – Car Purchase History Analysis (20%)

Write a module to read the purchase history file (i.e., car_sales.txt); then count and display the most frequent **FIVE** car brands that being sold in the log. The results **MUST** contain the **“top-5 frequent Car Brands and their sales count”**. You can return the results in any format (e.g., list, dict), not necessarily in dataframe. E.g., (Car Brand, count)

The purchase history file that we use for this assignment is the car sales file. The file looks something like the following: `Ford - - 2015 SUV Auto Black 26`

Each part of this file is described below.
- `Ford`: This is the car brand of one sales record, which means that for this sales record, the car that being sold is a Ford car. **(This is what you need to extract!)**
- `-`: The "hyphen" in the output indicates that the requested piece of information (user identity) is not available.
- `-`: The "hyphen" in the output indicates that the requested piece of information (user DOB) is not available.
- `2015` : The time that the car was being sold.
- `SUV` : This is the car type of the sale record, we have "SUV", "Passenger", "Hardtop" and so on.
- `Auto` : This means the car is in "Auto mode" or in "Manual mode".
- `Black` : This is the color of the car.
- `26` : This is the sales price of the car (26 means 26,000$).


In general, your module will contain (but not limited to) the following parts: 

1. A component that reads the log file
1. A function that extracts the car brand (e.g., Ford, Chevrolet, … ) from the file (**Hints**: the car brands are **the starting string to the first “-” of each row**. You may use **slicing** to extract the substring from text; and use map() to apply slicing on each row of the text file)
1. A component that counts the car sales amount and sort them descendingly.


In [4]:
import sys
print(f"Python version: {sys.version}")
print(f"Python executable: {sys.executable}")

!{sys.executable} -m pip install pyspark rouge-score

Python version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Python executable: c:\Users\User\AppData\Local\Programs\Python\Python311\python.exe



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# Setup: Set environment variables for PySpark on Windows
import os, sys

os.environ["JAVA_HOME"] = r"C:\Program Files\Microsoft\jdk-17.0.18.8-hotspot"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Set HADOOP_HOME for winutils on Windows
hadoop_dir = os.path.join(os.getcwd(), "hadoop")
bin_dir = os.path.join(hadoop_dir, "bin")
os.makedirs(bin_dir, exist_ok=True)
os.environ["HADOOP_HOME"] = hadoop_dir

# Add hadoop/bin to PATH so hadoop.dll can be found
os.environ["PATH"] = bin_dir + ";" + os.environ.get("PATH", "")

import urllib.request

# Download winutils.exe if not present
winutils_path = os.path.join(bin_dir, "winutils.exe")
if not os.path.exists(winutils_path):
    print("Downloading winutils.exe...")
    urllib.request.urlretrieve(
        "https://github.com/cdarlint/winutils/raw/master/hadoop-3.3.6/bin/winutils.exe",
        winutils_path)
    print("Done.")

# Download hadoop.dll if not present (required for file I/O on Windows)
hadoop_dll_path = os.path.join(bin_dir, "hadoop.dll")
if not os.path.exists(hadoop_dll_path):
    print("Downloading hadoop.dll...")
    urllib.request.urlretrieve(
        "https://github.com/cdarlint/winutils/raw/master/hadoop-3.3.6/bin/hadoop.dll",
        hadoop_dll_path)
    print("Done.")

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("DSAI4205_Assignment2") \
    .master("local[1]") \
    .config("spark.python.worker.reuse", "true") \
    .config("spark.python.worker.timeout", "120") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()
sc = spark.sparkContext
print("Spark session ready.")

Spark session ready.


In [6]:
# Question 1: Car Purchase History Analysis

from collections import Counter

# 1. Read the log file
with open("car_sales.txt", "r") as f:
    lines = f.readlines()

# 2. Extract car brand using map() and slicing
# The car brand is the starting string to the first "-" of each row
def extract_brand(line):
    return line[:line.index("-")].strip()

brands = list(map(extract_brand, lines))

# 3. Count car sales, sort descendingly, and get top 5
brand_counts = Counter(brands)
top_5 = brand_counts.most_common(5)

# Display results
print("Top-5 Frequent Car Brands and Their Sales Count:")
for brand, count in top_5:
    print(f"({brand}, {count})")

Top-5 Frequent Car Brands and Their Sales Count:
(Chevrolet, 177)
(Dodge, 173)
(Ford, 167)
(Mitsubishi, 146)
(Volkswagen, 131)


### Question 2 - Big Data Analysis for Student Plagiarism Detection (80%)

a)	Construct a Student Table as a PySpark DataFrame containing relevant student information as below. (The content is provided in the file student_data.txt, which can be found on Blackboard) (10%)

<font color="red">Task 2.1.1: Construct a Student Table as a PySpark DataFrame.</font>

In [7]:
# Task 2.1.1: Construct a Student Table as a PySpark DataFrame

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Define schema
schema = StructType([
    StructField("StudentID", StringType(), True),
    StructField("Name", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Gender", StringType(), True),
    StructField("Grade", StringType(), True),
    StructField("Year of Study", IntegerType(), True),
    StructField("Major", StringType(), True),
    StructField("Homework", StringType(), True),
    StructField("Past GPA", DoubleType(), True)
])

# Define 10 rows of data with mixed homework similarities
data = [
    ("12aB34", "Alice",  20, "F", "B+", 2, "CS",     "The sky is blue. I enjoy reading books. Cats are curious.",                           3.5),
    ("44rT32", "Daniel", 20, "M", "B",  2, "CS",     "Cats are curious. The sky is blue. I enjoy reading books.",                           3.4),
    ("05Xx21", "Nora",   21, "F", "A",  3, "CS",     "Algorithms are efficient. Data structures organize data. Python is popular.",         3.6),
    ("98Zx56", "Brian",  21, "M", "A-", 3, "Lit",    "I enjoy reading books. Cats are curious. The sky is blue.",                           3.7),
    ("75qQ12", "Faisal", 23, "M", "C+", 5, "Lit",    "Shakespeare wrote plays. Poetry uses metaphors. Literature reflects society.",        2.9),
    ("67Lm89", "Carla",  22, "F", "A",  4, "Bio",    "Photosynthesis needs sunlight. Leaves are green. Plants grow in soil.",               3.9),
    ("88Hg45", "Grace",  21, "F", "C",  3, "Bio",    "Leaves are green. Plants grow in soil. Photosynthesis needs sunlight.",               1.9),
    ("31Oo76", "Emily",  19, "F", "A-", 1, "Hist",   "The past shapes the future. Wars affect nations. People write history.",              3.8),
    ("22Ws90", "Henry",  20, "M", "B-", 2, "Hist",   "Trade routes shaped empires. The Renaissance sparked ideas. Monarchs ruled nations.", 3.2),
    ("59Kk38", "Jack",   22, "M", "B",  4, "EnvSci", "Climate change affects weather. Ecosystems are fragile. We must reduce emissions.",   3.3),
]

# Create DataFrame
student_df = spark.createDataFrame(data, schema)

# Cache and show the DataFrame
student_df = student_df.cache()
student_df.show(truncate=False)

+---------+------+---+------+-----+-------------+------+-----------------------------------------------------------------------------------+--------+
|StudentID|Name  |Age|Gender|Grade|Year of Study|Major |Homework                                                                           |Past GPA|
+---------+------+---+------+-----+-------------+------+-----------------------------------------------------------------------------------+--------+
|12aB34   |Alice |20 |F     |B+   |2            |CS    |The sky is blue. I enjoy reading books. Cats are curious.                          |3.5     |
|44rT32   |Daniel|20 |M     |B    |2            |CS    |Cats are curious. The sky is blue. I enjoy reading books.                          |3.4     |
|05Xx21   |Nora  |21 |F     |A    |3            |CS    |Algorithms are efficient. Data structures organize data. Python is popular.        |3.6     |
|98Zx56   |Brian |21 |M     |A-   |3            |Lit   |I enjoy reading books. Cats are curious. The

<font color="red">Task 2.1.2: Export the DataFrame to a JSON file.</font>

In [8]:
# Task 2.1.2: Export the DataFrame to a JSON file
student_df.write.mode("overwrite").json("student_data_output.json")
print("DataFrame exported to student_data_output.json")

DataFrame exported to student_data_output.json


b)	Compute Similarity Scores Among Students of the Same Major. (30%)
For students within the same major, compare their homework submissions and calculate a similarity score based on ROUGE-L metric.


In [9]:
from rouge_score import rouge_scorer
from pyspark.sql.functions import col, udf, collect_list, coalesce, array, lit
from pyspark.sql.types import ArrayType, FloatType, StringType

# Step 2.2.1: Tokenize Homework - split into sentences
def tokenize_homework(text):
    return [s.strip() for s in text.split(".") if s.strip()]

# Step 2.2.2: Define ROUGE-L similarity computation
# Returns the SUM of all pairwise ROUGE-L F1-scores
def compute_rouge_l(sentences_a, sentences_b):
    scorer_obj = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    total_score = 0.0
    for sent_a in sentences_a:
        for sent_b in sentences_b:
            score = scorer_obj.score(sent_a, sent_b)
            total_score += score['rougeL'].fmeasure
    return round(total_score, 4)

# Step 2.2.3: Use crossJoin to generate all student pairs, then filter for same major
student_a = student_df.select(
    col("StudentID").alias("ID_A"), col("Name").alias("Name_A"),
    col("Major").alias("Major_A"), col("Homework").alias("Homework_A")
)
student_b = student_df.select(
    col("StudentID").alias("ID_B"), col("Name").alias("Name_B"),
    col("Major").alias("Major_B"), col("Homework").alias("Homework_B")
)

# crossJoin produces the full cartesian product, then filter same major & different student
all_pairs = student_a.crossJoin(student_b) \
    .filter(col("Major_A") == col("Major_B")) \
    .filter(col("ID_A") != col("ID_B"))

# Step 2.2.4: Compute ROUGE-L scores using a UDF
@udf(FloatType())
def rouge_l_udf(hw_a, hw_b):
    sents_a = tokenize_homework(hw_a)
    sents_b = tokenize_homework(hw_b)
    return float(compute_rouge_l(sents_a, sents_b))

scored_pairs = all_pairs.withColumn("sim_score", rouge_l_udf(col("Homework_A"), col("Homework_B")))

# Step 2.2.5: Aggregate sim_names and sim_scores per student
sim_agg = scored_pairs.groupBy("ID_A").agg(
    collect_list("Name_B").alias("sim_names"),
    collect_list("sim_score").alias("sim_scores")
)

# Step 2.2.6: Join back to student_df and build result
comparison_df = student_df.join(sim_agg, student_df["StudentID"] == sim_agg["ID_A"], "left") \
    .drop("ID_A", "StudentID")

# Handle students with no same-major peers (e.g., Jack in EnvSci) - replace nulls with empty arrays
comparison_df = comparison_df.withColumn("sim_names", coalesce(col("sim_names"), array().cast(ArrayType(StringType())))) \
    .withColumn("sim_scores", coalesce(col("sim_scores"), array().cast(ArrayType(FloatType()))))

comparison_df = comparison_df.select(
    "Name", "Age", "Gender", "Grade", "Year of Study",
    "Major", "Homework", "Past GPA", "sim_names", "sim_scores"
)

# Sort by Major then Name, cache, and show
comparison_df = comparison_df.orderBy("Major", "Name")
comparison_df = comparison_df.cache()
comparison_df.show(truncate=False)

+------+---+------+-----+-------------+------+-----------------------------------------------------------------------------------+--------+---------------+--------------+
|Name  |Age|Gender|Grade|Year of Study|Major |Homework                                                                           |Past GPA|sim_names      |sim_scores    |
+------+---+------+-----+-------------+------+-----------------------------------------------------------------------------------+--------+---------------+--------------+
|Carla |22 |F     |A    |4            |Bio   |Photosynthesis needs sunlight. Leaves are green. Plants grow in soil.              |3.9     |[Grace]        |[3.0]         |
|Grace |21 |F     |C    |3            |Bio   |Leaves are green. Plants grow in soil. Photosynthesis needs sunlight.              |1.9     |[Carla]        |[3.0]         |
|Alice |20 |F     |B+   |2            |CS    |The sky is blue. I enjoy reading books. Cats are curious.                          |3.5     |[Nora,

c)	Identify Plagiarism Cases: (20%)

Assume that a similarity score of 3 (i.e., sim_scores = 3) indicates plagiarism between students of the same major.
Implement a DataFrame querying system to answer the following questions:
- "Who plagiarized in Major X?" (x mean a particular Major, e.g., Bio)
- "How many students plagiarized in each major?"
- Notes:
    - (15%) will be awarded for correctly implementing the query system.
    - (5%) will be awarded if the system can handle different phrasings or patterns of similar questions 
    (e.g., "Who copied in Major X?", "List students who cheated in CS").


In [10]:
import re
from pyspark.sql.functions import col, explode, arrays_zip, count

def answer_question(question, df):
    """Query system for plagiarism detection.
    df has per-student rows with sim_names and sim_scores arrays.
    """
    # Explode into pairwise rows and filter for plagiarism (score >= 3)
    exploded_df = df.select(
        col("Name").alias("StudentA"), col("Major"), col("`Past GPA`"),
        explode(arrays_zip("sim_names", "sim_scores")).alias("sim")
    ).select(
        "StudentA", "Major", "`Past GPA`",
        col("sim.sim_names").alias("StudentB"),
        col("sim.sim_scores").alias("SimScore")
    )

    plagiarism_df = exploded_df.filter(col("SimScore") >= 3)
    # De-duplicate pairs: keep only where StudentA < StudentB alphabetically
    plagiarism_pairs = plagiarism_df.filter(col("StudentA") < col("StudentB"))

    question_lower = question.lower()

    # Pattern: "Who plagiarized/copied/cheated in [major]?"
    who_pattern = r"(?:who\s+(?:plagiarized|plagiarised|copied|cheated)|list\s+students\s+who\s+(?:plagiarized|plagiarised|copied|cheated))\s+in\s+(?:the\s+)?(?:major\s+)?(\w+)"
    who_match = re.search(who_pattern, question_lower)

    if who_match:
        major = who_match.group(1)
        result = plagiarism_pairs.filter(
            col("Major").rlike("(?i)^" + major + "$")
        ).select("StudentA", "StudentB", "Major", "SimScore")
        return result

    # Pattern: "How many students plagiarized in each major?"
    count_pattern = r"how many\s+students\s+(?:plagiarized|plagiarised|copied|cheated)\s+in\s+each\s+major"
    if re.search(count_pattern, question_lower):
        result = plagiarism_pairs.groupBy("Major").agg(count("*").alias("Plagiarism_Count"))
        return result

    # Handling unknown questions
    print("Sorry, I could not understand the question.")
    return plagiarism_pairs.limit(0)

# Test with sample questions matching the PDF examples
question1 = "Who plagiarized in the Major bio?"
print(f"Question: {question1}")
print("Answer:")
answer_question(question1, comparison_df).show(truncate=False)

print()

question2 = "Who plagiarized in the Major EnvSci?"
print(f"Question: {question2}")
print("Answer:")
answer_question(question2, comparison_df).show(truncate=False)

print()

question3 = "Who plagiarized in the Major hist?"
print(f"Question: {question3}")
print("Answer:")
answer_question(question3, comparison_df).show(truncate=False)

print()

question4 = "How many students plagiarized in each major?"
print(f"Question: {question4}")
print("Answer:")
answer_question(question4, comparison_df).show(truncate=False)

print()

# Additional test: different phrasing
question5 = "List students who cheated in CS"
print(f"Question: {question5}")
print("Answer:")
answer_question(question5, comparison_df).show(truncate=False)

Question: Who plagiarized in the Major bio?
Answer:
+--------+--------+-----+--------+
|StudentA|StudentB|Major|SimScore|
+--------+--------+-----+--------+
|Carla   |Grace   |Bio  |3.0     |
+--------+--------+-----+--------+


Question: Who plagiarized in the Major EnvSci?
Answer:
+--------+--------+-----+--------+
|StudentA|StudentB|Major|SimScore|
+--------+--------+-----+--------+
+--------+--------+-----+--------+


Question: Who plagiarized in the Major hist?
Answer:
+--------+--------+-----+--------+
|StudentA|StudentB|Major|SimScore|
+--------+--------+-----+--------+
+--------+--------+-----+--------+


Question: How many students plagiarized in each major?
Answer:
+-----+----------------+
|Major|Plagiarism_Count|
+-----+----------------+
|Bio  |1               |
|CS   |1               |
+-----+----------------+


Question: List students who cheated in CS
Answer:
+--------+--------+-----+--------+
|StudentA|StudentB|Major|SimScore|
+--------+--------+-----+--------+
|Alice   

d)	Determine Copying Relationships Based on GPA (20%)
- If the GPA difference ≤ 0.3, classify the relationship as "copy each other".
- If the GPA difference > 0.3, classify as a "copee-copier" relationship: 
    - The student with the higher GPA is the copee.
    - The student with the lower GPA is the copier.


In [11]:
from pyspark.sql.functions import col, when, abs as spark_abs, lit, explode, arrays_zip

# Step 2.4.1: Explode comparison_df to get pairwise plagiarism pairs
exploded_df = comparison_df.select(
    col("Name").alias("StudentA"), col("Major"), col("`Past GPA`").alias("GPA_A"),
    explode(arrays_zip("sim_names", "sim_scores")).alias("sim")
).select(
    "StudentA", "Major", "GPA_A",
    col("sim.sim_names").alias("StudentB"),
    col("sim.sim_scores").alias("SimScore")
)

# Step 2.4.2: Filter for plagiarism cases (score >= 3) and de-duplicate pairs
plagiarism_pairs = exploded_df.filter(col("SimScore") >= 3) \
                              .filter(col("StudentA") < col("StudentB"))

# Step 2.4.3: Join to get GPA of student B
gpa_lookup = comparison_df.select(col("Name"), col("`Past GPA`").alias("GPA_B")).distinct()
pairs_with_gpa = plagiarism_pairs.join(gpa_lookup, plagiarism_pairs["StudentB"] == gpa_lookup["Name"]) \
    .drop("Name")

# Step 2.4.4: Reorder so higher-GPA student is StudentA
pairs_ordered = pairs_with_gpa.select(
    when(col("GPA_A") >= col("GPA_B"), col("StudentA")).otherwise(col("StudentB")).alias("StudentA"),
    when(col("GPA_A") >= col("GPA_B"), col("GPA_A")).otherwise(col("GPA_B")).alias("GPA_A"),
    when(col("GPA_A") >= col("GPA_B"), col("StudentB")).otherwise(col("StudentA")).alias("StudentB"),
    when(col("GPA_A") >= col("GPA_B"), col("GPA_B")).otherwise(col("GPA_A")).alias("GPA_B"),
    col("Major"), col("SimScore")
)

# Step 2.4.5: Compare GPA and define relationship type
result_df = pairs_ordered.withColumn(
    "Relation",
    when(spark_abs(col("GPA_A") - col("GPA_B")) <= 0.3, lit("copy each other"))
    .otherwise(lit("copee-copier"))
)

# Step 2.4.6: Show results matching PDF format
result_df.select("StudentA", "GPA_A", "StudentB", "GPA_B", "Major", "SimScore", "Relation") \
    .orderBy("Major") \
    .show(truncate=False)

+--------+-----+--------+-----+-----+--------+---------------+
|StudentA|GPA_A|StudentB|GPA_B|Major|SimScore|Relation       |
+--------+-----+--------+-----+-----+--------+---------------+
|Carla   |3.9  |Grace   |1.9  |Bio  |3.0     |copee-copier   |
|Alice   |3.5  |Daniel  |3.4  |CS   |3.0     |copy each other|
+--------+-----+--------+-----+-----+--------+---------------+



**You've reached the end of the assignment. Don't forget to upload it to Blackboard.**